# JFMO EN-RU

Train model with datasets:
- RU titles: [Movie plots from Wikipedia in Russian](https://www.kaggle.com/datasets/maksimpotorochin/movie-plots-from-wikipedia-in-russian) + [Kion](https://ods.ai/competitions/competition-recsys-21/data) + [IMDb](https://developer.imdb.com/non-commercial-datasets/)
- EN titles: [Full TMDB Movies Dataset 2024 (1M Movies)](https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies) + [IMDb](https://developer.imdb.com/non-commercial-datasets/)

Inspared by [Language Identification for Texts Written in
Transliteration](https://ceur-ws.org/Vol-871/paper_2.pdf).

## Imports

In [1]:
!pip install -q transliterate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.8 MB/s eta 0:00:00


In [2]:
import os
import math

import pickle
import transliterate

from tqdm import tqdm
from google.colab import files
import pandas as pd
from collections import defaultdict, Counter

## Dataset

### Preprocessing

#### Download

In [3]:
!rm -rf ./data sample_data

os.environ['KAGGLE_USERNAME'] = ''
os.environ['KAGGLE_KEY'] = ''

Download RU Dataset 1: [Movie plots from Wikipedia in Russian](https://www.kaggle.com/datasets/maksimpotorochin/movie-plots-from-wikipedia-in-russian)

In [4]:
!kaggle datasets download maksimpotorochin/movie-plots-from-wikipedia-in-russian

!unzip -q movie-plots-from-wikipedia-in-russian.zip -d ./data

!rm -rf movie-plots-from-wikipedia-in-russian.zip

Dataset URL: https://www.kaggle.com/datasets/maksimpotorochin/movie-plots-from-wikipedia-in-russian
License(s): CC-BY-SA-3.0
  0% 0.00/41.9M [00:00<?, ?B/s]
100% 41.9M/41.9M [00:00<00:00, 674MB/s]


Download RU Dataset 2: [Kion](https://ods.ai/competitions/competition-recsys-21/data)

In [5]:
!wget https://storage.yandexcloud.net/datasouls-ods/materials/f90231b6/items.csv -P ./data

--2025-11-01 12:18:27--  https://storage.yandexcloud.net/datasouls-ods/materials/f90231b6/items.csv
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31836561 (30M) [text/csv]
Saving to: ‘./data/items.csv’

items.csv           100%[===================>]  30.36M  16.2MB/s    in 1.9s    

2025-11-01 12:18:30 (16.2 MB/s) - ‘./data/items.csv’ saved [31836561/31836561]



Download IMDB RU + EN: [IMDb](https://developer.imdb.com/non-commercial-datasets/)

In [6]:
!wget https://datasets.imdbws.com/title.akas.tsv.gz -P ./data
!gunzip ./data/title.akas.tsv.gz

--2025-11-01 12:18:30--  https://datasets.imdbws.com/title.akas.tsv.gz
Resolving datasets.imdbws.com (datasets.imdbws.com)... 13.249.98.73, 13.249.98.61, 13.249.98.35, ...
Connecting to datasets.imdbws.com (datasets.imdbws.com)|13.249.98.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 467082077 (445M) [binary/octet-stream]
Saving to: ‘./data/title.akas.tsv.gz’

title.akas.tsv.gz   100%[===================>] 445.44M   230MB/s    in 1.9s    

2025-11-01 12:18:32 (230 MB/s) - ‘./data/title.akas.tsv.gz’ saved [467082077/467082077]



Download EN Dataset: [Full TMDB Movies Dataset 2024 (1M Movies)](https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies)

In [7]:
!kaggle datasets download 'asaniczka/tmdb-movies-dataset-2023-930k-movies'

!unzip -q 'tmdb-movies-dataset-2023-930k-movies'.zip -d ./data

!rm -rf 'tmdb-movies-dataset-2023-930k-movies'.zip

Dataset URL: https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies
License(s): ODC Attribution License (ODC-By)
 92% 210M/228M [00:00<00:00, 575MB/s] 
100% 228M/228M [00:00<00:00, 582MB/s]


#### Read

RU Dataset 1: [Movie plots from Wikipedia in Russian](https://www.kaggle.com/datasets/maksimpotorochin/movie-plots-from-wikipedia-in-russian)

In [8]:
df_ru1 = pd.read_csv('data/films_data.csv')
print("RU Dataset 1 (films_data.csv):")
print(f"  Columns: {df_ru1.columns.tolist()}")
print(f"  Rows: {len(df_ru1)}")

RU Dataset 1 (films_data.csv):
  Columns: ['title', 'type', 'genre', 'imdb_rating', 'summary', 'plot']
  Rows: 45812


In [9]:
print(df_ru1.columns.tolist())

print(df_ru1.head())

['title', 'type', 'genre', 'imdb_rating', 'summary', 'plot']
                           title  type                  genre  imdb_rating  \
0                      ? (фильм)  film    драматический фильм          7.0   
1     ...а пятый всадник – Страх  film  драма военный артхаус          7.2   
2  …и передайте привет ласточкам  film          драма военный          6.8   
3       «Чудотворец» из Бирюлёва  film           игровое кино          NaN   
4           (Не)идеальные роботы  film     комедия фантастика          5.4   

                                             summary  \
0  «?» (индон. Tanda Tanya) — индонезийский худож...   
1  «…а пятый всадник — Страх» (чеш. …a páty jezde...   
2  «…и передайте привет ласточкам» — (чеш. ...a p...   
3  «Чудотворец из Бирюлёва» — советский короткоме...   
4  «(Не)идеальные роботы» (англ. Robots) — художе...   

                                                plot  
0  Основная тема фильма — межрелигиозные отношени...  
1  Прага во время немец

RU Dataset 2: [Kion](https://ods.ai/competitions/competition-recsys-21/data)

In [10]:
df_ru2 = pd.read_csv('data/items.csv', encoding='utf-8')
print("\nRU Dataset 2 (items.csv):")
print(f"  Columns: {df_ru2.columns.tolist()}")
print(f"  Rows: {len(df_ru2)}")


RU Dataset 2 (items.csv):
  Columns: ['item_id', 'content_type', 'title', 'title_orig', 'release_year', 'genres', 'countries', 'for_kids', 'age_rating', 'studios', 'directors', 'actors', 'description', 'keywords']
  Rows: 15963


In [11]:
print(df_ru2.columns.tolist())

print(df_ru2.head())

['item_id', 'content_type', 'title', 'title_orig', 'release_year', 'genres', 'countries', 'for_kids', 'age_rating', 'studios', 'directors', 'actors', 'description', 'keywords']
   item_id content_type                 title      title_orig  release_year  \
0    10711         film        Поговори с ней  Hable con ella        2002.0   
1     2508         film           Голые перцы    Search Party        2014.0   
2    10716         film      Тактическая сила  Tactical Force        2011.0   
3     7868         film                45 лет        45 Years        2015.0   
4    16268         film  Все решает мгновение             NaN        1978.0   

                                             genres       countries  for_kids  \
0           драмы, зарубежные, детективы, мелодрамы         Испания       NaN   
1                  зарубежные, приключения, комедии             США       NaN   
2  криминал, зарубежные, триллеры, боевики, комедии          Канада       NaN   
3                      д

IMDB RU + EN: [IMDb](https://developer.imdb.com/non-commercial-datasets/)

In [12]:
chunks_ru = []
chunks_en = []
chunk_size = 100000

for chunk in tqdm(
    pd.read_csv('data/title.akas.tsv', sep='\t', chunksize=chunk_size, low_memory=False),
    desc="Processing IMDB chunks"
):
    chunk_ru = chunk[
        (chunk['region'] == 'RU') |
        (chunk['language'] == 'ru')
    ]
    if len(chunk_ru) > 0:
        chunks_ru.append(chunk_ru)

    chunk_en = chunk[
        (chunk['region'] == 'US') |
        (chunk['region'] == 'GB') |
        (chunk['language'] == 'en')
    ]
    if len(chunk_en) > 0:
        chunks_en.append(chunk_en)

Processing IMDB chunks: 537it [02:08,  4.18it/s]


EN Dataset: [Full TMDB Movies Dataset 2024 (1M Movies)](https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies)

In [13]:
df_en = pd.read_csv('data/TMDB_movie_dataset_v11.csv')
print("\nEN Dataset (TMDB):")
print(f"  Columns: {df_en.columns.tolist()}")
print(f"  Rows: {len(df_en)}")


EN Dataset (TMDB):
  Columns: ['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords']
  Rows: 1313491


In [14]:
print(df_en.columns.tolist())

print(df_en.head())

['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords']
       id            title  vote_average  vote_count    status release_date  \
0   27205        Inception         8.364       34495  Released   2010-07-15   
1  157336     Interstellar         8.417       32571  Released   2014-11-05   
2     155  The Dark Knight         8.512       30619  Released   2008-07-16   
3   19995           Avatar         7.573       29815  Released   2009-12-15   
4   24428     The Avengers         7.710       29166  Released   2012-04-25   

      revenue  runtime  adult                     backdrop_path  ...  \
0   825532764      148  False  /8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg  ...   
1   701729206      169  False  /pbrkL804c8yAv3z

#### Extract

RU

In [15]:
ru_titles_1 = df_ru1['title'].dropna().tolist()
ru_types_1 = df_ru1['type'].dropna().tolist()
print(f"\nRU titles from films_data.csv: {len(ru_titles_1)}")


RU titles from films_data.csv: 45812


In [16]:
ru_titles_2 = df_ru2['title'].dropna().tolist()
ru_types_2 = df_ru2['content_type'].dropna().tolist()
print(f"RU titles from items.csv: {len(ru_titles_2)}")

RU titles from items.csv: 15963


In [17]:
df_imdb_ru = pd.concat(chunks_ru, ignore_index=True) if chunks_ru else pd.DataFrame()
ru_titles_3 = df_imdb_ru['title'].dropna().tolist()
ru_types_3 = [None] * len(ru_titles_3)
print(f"RU titles from IMDB: {len(ru_titles_3)}")

RU titles from IMDB: 167379


Normalize

In [18]:
def normalize_type(type_val):
    if type_val is None:
        return None
    type_str = str(type_val).lower().strip()
    if 'film' in type_str or 'movie' in type_str:
        return 'movie'
    elif 'series' in type_str:
        return 'tv series'
    return type_val

Combine RU titles (remove duplicates)

In [19]:
all_ru_titles = ru_titles_1 + ru_titles_2 + ru_titles_3
all_ru_types = ru_types_1 + ru_types_2 + ru_types_3

# Remove duplicates while preserving order
ru_titles_unique = []
ru_types_unique = []
seen = set()

for title, type_val in zip(all_ru_titles, all_ru_types):
    title_lower = str(title).lower().strip()
    if title_lower not in seen:
        seen.add(title_lower)
        ru_titles_unique.append(title)
        ru_types_unique.append(normalize_type(type_val))

print(f"\nUnique RU titles: {len(ru_titles_unique)}")


Unique RU titles: 165970


EN

In [20]:
en_titles_1 = df_en['title'].dropna().tolist()
en_types_1 = ['movie'] * len(en_titles_1)
print(f"\nEN titles from TMDB: {len(en_titles_1)}")


EN titles from TMDB: 1313475


In [21]:
df_imdb_en = pd.concat(chunks_en, ignore_index=True) if chunks_en else pd.DataFrame()
en_titles_2 = df_imdb_en['title'].dropna().tolist()
en_types_2 = [None] * len(en_titles_2)
print(f"EN titles from IMDB: {len(en_titles_2)}")

EN titles from IMDB: 2844357


In [22]:
all_en_titles = en_titles_1 + en_titles_2
all_en_types = en_types_1 + en_types_2

# Remove duplicates
en_titles_unique = []
en_types_unique = []
seen_en = set()

for title, type_val in zip(all_en_titles, all_en_types):
    title_lower = str(title).lower().strip()
    if title_lower not in seen_en:
        seen_en.add(title_lower)
        en_titles_unique.append(title)
        en_types_unique.append(type_val)

print(f"\nUnique EN titles: {len(en_titles_unique)}")


Unique EN titles: 2399851


#### Transliterate

In [23]:
print(f"Transliterating {len(ru_titles_unique)} titles...")

ru_titles_translit = []
errors_ru = 0

Transliterating 165970 titles...


In [24]:
for text in tqdm(ru_titles_unique, desc="Transliterating"):
    try:
        translit = transliterate.translit(str(text), 'ru', reversed=True)
        ru_titles_translit.append(translit)
    except Exception as e:
        errors_ru += 1
        ru_titles_translit.append(None)

print(f"\nTransliteration errors: {errors_ru}")

Transliterating: 100%|██████████| 165970/165970 [00:09<00:00, 18098.51it/s]

Transliteration errors: 0


### Create train datasets

RU

In [25]:
df_ru_final = pd.DataFrame({
    'title': ru_titles_unique[:len(ru_titles_translit)],
    'language': 'ru',
    'title_translit': ru_titles_translit,
    'type': ru_types_unique[:len(ru_titles_translit)]
})

In [26]:
df_ru_final = df_ru_final[df_ru_final['title_translit'].notna()]
print(f"\nRU final (after removing failed): {len(df_ru_final)} rows")


RU final (after removing failed): 165970 rows


EN

In [27]:
df_en_final = pd.DataFrame({
    'title': en_titles_unique,
    'language': 'en',
    'title_translit': None,
    'type': en_types_unique
})

print(f"EN final: {len(df_en_final)} rows")

EN final: 2399851 rows


Combine

In [28]:
train_df = pd.concat([df_ru_final, df_en_final], ignore_index=True)

print(f"\nTrain dataset combined: {len(train_df)} rows")
print(f"  - RU: {len(df_ru_final)} rows")
print(f"  - EN: {len(df_en_final)} rows")
print(f"\nColumns: {train_df.columns.tolist()}")
print("\nSample rows:")
print(train_df.head(10))


Train dataset combined: 2565821 rows
  - RU: 165970 rows
  - EN: 2399851 rows

Columns: ['title', 'language', 'title_translit', 'type']

Sample rows:
                           title language                  title_translit  \
0                      ? (фильм)       ru                       ? (fil'm)   
1     ...а пятый всадник – Страх       ru     ...a pjatyj vsadnik – Strah   
2  …и передайте привет ласточкам       ru  …i peredajte privet lastochkam   
3       «Чудотворец» из Бирюлёва       ru     «Chudotvorets» iz Birjuleva   
4           (Не)идеальные роботы       ru            (Ne)ideal'nye roboty   
5                          0-41*       ru                           0-41*   
6       1 (документальный фильм)       ru        1 (dokumental'nyj fil'm)   
7                    1 % (фильм)       ru                     1 % (fil'm)   
8                 1 миля до тебя       ru                1 milja do tebja   
9               1 Night in China       ru                1 Night in China   

 

In [29]:
train_df.to_csv('media_transliterated.csv', index=False, encoding='utf-8')
print(f"\n💾 Dataset saved: media_transliterated.csv ({len(train_df)} rows)")


💾 Dataset saved: media_transliterated.csv (2565821 rows)


## Model

In [30]:
class NgramModel:
    def __init__(self, n=3):
        self.n = n
        self.ngrams = defaultdict(Counter)

    def train(self, texts):
        from tqdm import tqdm

        for text in tqdm(texts, desc="Training n-grams"):
            text = text.lower() + ' '

            for i in range(len(text) - self.n):
                context = text[i:i+self.n]
                next_char = text[i+self.n]
                self.ngrams[context][next_char] += 1

    def probability(self, text):
        text = text.lower() + ' '
        log_prob = 0.0

        for i in range(len(text) - self.n):
            context = text[i:i+self.n]
            next_char = text[i+self.n]

            count_next = self.ngrams[context][next_char]
            count_total = sum(self.ngrams[context].values())

            if count_total > 0:
                prob = (count_next + 1) / (count_total + 100)
                log_prob += math.log(prob)
            else:
                log_prob += math.log(0.001)

        return log_prob / len(text) if len(text) > 0 else -1000

    def save(self, filepath):
        with open(filepath, 'wb') as f:
            pickle.dump({'n': self.n, 'ngrams': dict(self.ngrams)}, f)
        print(f"Model saved to: {filepath}")

    def load(self, filepath):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
            self.n = data['n']
            self.ngrams = defaultdict(Counter, data['ngrams'])

Define

In [40]:
model_ru = NgramModel(n=3)
model_en = NgramModel(n=3)

## Train

RU

In [41]:
ru_texts = train_df[train_df['language'] == 'ru']['title_translit'].dropna().tolist()
print(f"Training RU model with {len(ru_texts)} texts...")
model_ru.train(ru_texts)

Training RU model with 165970 texts...


Training n-grams: 100%|██████████| 165970/165970 [00:02<00:00, 75550.51it/s]


EN

In [42]:
en_texts = train_df[train_df['language'] == 'en']['title'].dropna().tolist()
print(f"Training EN model with {len(en_texts)} texts...")
model_en.train(en_texts)

Training EN model with 2399851 texts...


Training n-grams: 100%|██████████| 2399851/2399851 [00:45<00:00, 53159.97it/s]


## Save model

In [43]:
model_ru.save('jfmo_russian_model.pkl')
model_en.save('jfmo_english_model.pkl')

Model saved to: jfmo_russian_model.pkl
Model saved to: jfmo_english_model.pkl


## Test

In [44]:
test_cases = [
    # Russian films - Classics
    ("Brat", True),
    ("Brat 2", True),
    ("Stalker", True),
    ("Solaris", True),
    ("Moskva slezam ne verit", True),
    ("Leviafan", True),
    ("Nochnoy dozor", True),
    ("Dnevnoy dozor", True),
    ("Ironia sudby", True),
    ("Brilliantovaya ruka", True),
    ("Voyna i mir", True),
    ("Ostrov", True),
    ("Vozvrashchenie", True),
    ("Beloe solntse pustyni", True),
    ("Operatsiya Y", True),
    ("Kavkazskaya plennitsa", True),
    ("Ivan Vasilevich", True),
    ("Sluzhebnyy roman", True),
    ("Zerkalo", True),
    ("Andrey Rublev", True),

    # Russian films - Tarkovsky
    ("Ivanovo detstvo", True),
    ("Nostalghiya", True),
    ("Offret", True),
    ("Katok i skripka", True),

    # Russian films - Soviet Era
    ("Ballada o soldate", True),
    ("Letyat zhuravli", True),
    ("Chelovek s kinoapparatom", True),
    ("Chapayev", True),
    ("Aleksandr Nevskiy", True),
    ("Ivan Groznyy", True),
    ("Bronenosets Potemkin", True),
    ("Zemlya", True),
    ("Novaya Moskva", True),
    ("Tsirk", True),
    ("Volga Volga", True),
    ("Vesna", True),
    ("Kubanskie kazaki", True),
    ("Karnavalnaya noch", True),
    ("Vysota", True),
    ("Devyat dney odnogo goda", True),

    # Russian films - Modern
    ("Gruz 200", True),
    ("4", True),
    ("Koktebel", True),
    ("Prostye veshchi", True),
    ("Zvezda", True),
    ("Zhit", True),
    ("Durak", True),
    ("Nelюbov", True),
    ("Rai", True),
    ("Iskupleniye", True),
    ("Voyna Anny", True),
    ("Podolskie kursanty", True),
    ("Zoya", True),
    ("Kalashnikov", True),
    ("Legenda o Kolovrate", True),
    ("Tobol", True),
    ("Tanki", True),
    ("T-34", True),
    ("Балканский rubezh", True),
    ("Devyataev", True),
    ("Krasnyy prizrak", True),
    ("Krym", True),
    ("Ogon", True),

    # Russian films - Comedy
    ("Dzhentlemeny udachi", True),
    ("Beregis avtomobilya", True),
    ("Pokrovskie vorota", True),
    ("Osenniy marafon", True),
    ("Dvеnadtsat stulyev", True),
    ("Zolotoy telenok", True),
    ("Sobachye serdtse", True),
    ("Formula lyubvi", True),
    ("Tot samyy Myunkhgauzen", True),
    ("Obyknovennoe chudo", True),
    ("Ubit drakona", True),
    ("Gorod Zero", True),
    ("Assa", True),
    ("Igla", True),
    ("Taksi blyuz", True),
    ("Kholodnoe leto pyatdesyat tretyego", True),
    ("Malenkaya Vera", True),
    ("Interdevochka", True),

    # Russian films - Genre
    ("Osobennosti natsionalnoy okhoty", True),
    ("Osobennosti natsionalnoy rybalki", True),
    ("Den radio", True),
    ("Den vyborov", True),
    ("Gorko", True),
    ("Elki", True),
    ("Samyy luchshiy den", True),
    ("Prityazhenie", True),
    ("Dvizhenie vverkh", True),
    ("Led", True),
    ("Trener", True),
    ("Salyut-7", True),
    ("Vremya pervykh", True),
    ("Viking", True),
    ("Legenda nomer 17", True),
    ("Spasti Leningrad", True),
    ("Soyuz spaseniya", True),

    # Russian films - Sci-Fi/Fantasy
    ("Metro", True),
    ("Obitaemyy ostrov", True),
    ("Turetskiy gambit", True),
    ("Statskiy sovetnik", True),
    ("Admiral", True),
    ("Sibirskiy tsiryulnik", True),
    ("Musulmanin", True),
    ("Dom durakov", True),
    ("Gruz", True),
    ("Kray", True),
    ("Ovsyanki", True),

    # Russian films - Recent
    ("Podbrosy", True),
    ("Vremennye trudnosti", True),
    ("Odnoklassniki", True),
    ("Gorko 2", True),
    ("Gorko 3", True),
    ("Vezuchiy sluchay", True),
    ("Elki 1914", True),
    ("Elki 2", True),
    ("Elki 3", True),
    ("Elki lokhmatye", True),
    ("Elki poslednie", True),

    # Russian TV Series
    ("Politseyskiy s Rublevki", True),
    ("Interny", True),
    ("Kukhnya", True),
    ("Realnye patsany", True),
    ("Fizruk", True),
    ("Papiny dochki", True),
    ("Voroniny", True),
    ("Univer", True),
    ("Univer Novaya obshchaga", True),
    ("SashaTanya", True),
    ("Olga", True),
    ("Izmeny", True),
    ("Mazhor", True),
    ("Mosgaz", True),
    ("Gogol", True),
    ("Soderzhanki", True),
    ("Fartsa", True),
    ("Trotskiy", True),
    ("Godunov", True),
    ("Zuleykha otkryvaet glaza", True),
    ("Epidemiya", True),
    ("Magomaev", True),
    ("Chiki", True),
    ("Slovo patsana Krov na asfalte", True),
    ("Master i Margarita", True),
    ("Demon revolyutsii", True),
    ("Nyukhach", True),
    ("Sled", True),
    ("Uboynaya sila", True),
    ("Glukhar", True),
    ("Tayny sledstviya", True),
    ("Sledstvie veli", True),
    ("Ulitsy razbitykh fonarey", True),
    ("Banditskiy Peterburg", True),
    ("Kamenskaya", True),
    ("Brat za brata", True),
    ("Likvidatsiya", True),
    ("Mesto vstrechi izmenit nelzya", True),
    ("Semnadtsat mgnoveniy vesny", True),
    ("Ognyom i mechom", True),
    ("Shtrafbat", True),
    ("Diversant", True),
    ("Rzhev", True),
    ("Silnee ognya", True),

    # English films - Classics
    ("The Matrix", False),
    ("Fight Club", False),
    ("Inception", False),
    ("The Godfather", False),
    ("Pulp Fiction", False),
    ("The Shawshank Redemption", False),
    ("Forrest Gump", False),
    ("Star Wars", False),
    ("Interstellar", False),
    ("The Dark Knight", False),
    ("Titanic", False),
    ("Avatar", False),
    ("Gladiator", False),
    ("Saving Private Ryan", False),
    ("Schindler's List", False),

    # English films - More classics
    ("The Green Mile", False),
    ("The Departed", False),
    ("The Prestige", False),
    ("Memento", False),
    ("The Usual Suspects", False),
    ("American History X", False),
    ("The Pianist", False),
    ("The Lion King", False),
    ("Back to the Future", False),
    ("The Silence of the Lambs", False),
    ("Seven", False),
    ("Casino", False),
    ("Heat", False),
    ("Goodfellas", False),
    ("The Untouchables", False),

    # English films - Marvel/DC
    ("Iron Man", False),
    ("The Avengers", False),
    ("Captain America", False),
    ("Thor", False),
    ("Black Panther", False),
    ("Spider-Man", False),
    ("Guardians of the Galaxy", False),
    ("Doctor Strange", False),
    ("Ant-Man", False),
    ("The Batman", False),
    ("Wonder Woman", False),
    ("Aquaman", False),
    ("Suicide Squad", False),
    ("Justice League", False),
    ("Man of Steel", False),

    # English films - Sci-Fi
    ("Blade Runner", False),
    ("The Terminator", False),
    ("Alien", False),
    ("The Fifth Element", False),
    ("Minority Report", False),
    ("Total Recall", False),
    ("Edge of Tomorrow", False),
    ("District 9", False),
    ("Arrival", False),
    ("Ex Machina", False),
    ("Moon", False),
    ("Gravity", False),
    ("The Martian", False),
    ("Prometheus", False),
    ("Dune", False),

    # English films - Horror
    ("The Exorcist", False),
    ("The Shining", False),
    ("Psycho", False),
    ("Jaws", False),
    ("Halloween", False),
    ("A Nightmare on Elm Street", False),
    ("The Ring", False),
    ("The Conjuring", False),
    ("It", False),
    ("Get Out", False),
    ("Hereditary", False),
    ("The Witch", False),
    ("Midsommar", False),
    ("A Quiet Place", False),
    ("The Babadook", False),

    # English films - Animation
    ("Toy Story", False),
    ("Finding Nemo", False),
    ("Shrek", False),
    ("How to Train Your Dragon", False),
    ("Frozen", False),
    ("Moana", False),
    ("Coco", False),
    ("Inside Out", False),
    ("Up", False),
    ("WALL-E", False),
    ("Ratatouille", False),
    ("Monsters Inc", False),
    ("The Incredibles", False),
    ("Zootopia", False),
    ("Big Hero 6", False),

    # English films - Action
    ("Die Hard", False),
    ("Lethal Weapon", False),
    ("Mad Max Fury Road", False),
    ("John Wick", False),
    ("Mission Impossible", False),
    ("The Bourne Identity", False),
    ("Casino Royale", False),
    ("Skyfall", False),
    ("Spectre", False),
    ("Fast and Furious", False),
    ("The Raid", False),
    ("Taken", False),
    ("300", False),
    ("Kill Bill", False),
    ("Sin City", False),

    # English films - Drama
    ("12 Years a Slave", False),
    ("Moonlight", False),
    ("Birdman", False),
    ("Whiplash", False),
    ("La La Land", False),
    ("Manchester by the Sea", False),
    ("Room", False),
    ("The Revenant", False),
    ("Spotlight", False),
    ("Boyhood", False),
    ("Her", False),
    ("12 Angry Men", False),
    ("A Beautiful Mind", False),
    ("Rain Man", False),
    ("Good Will Hunting", False),

    # English films - Comedy
    ("The Hangover", False),
    ("Superbad", False),
    ("Bridesmaids", False),
    ("Step Brothers", False),
    ("Anchorman", False),
    ("Dumb and Dumber", False),
    ("Groundhog Day", False),
    ("The Big Lebowski", False),
    ("Monty Python and the Holy Grail", False),
    ("Airplane", False),
    ("Blazing Saddles", False),
    ("Some Like It Hot", False),
    ("Ferris Bueller's Day Off", False),
    ("Caddyshack", False),
    ("Animal House", False),

    # EDGE CASES - Ambiguous
    ("Sputnik", True),
    ("Matrix", False),
    ("Avatar 2", False),
    ("Cheburashka", True),
    ("Pokémon", False),
    ("Rocky", False),
    ("Rambo", False),
    ("Tarzan", False),
    ("Dracula", False),
    ("Frankenstein", False),
    ("Mulan", False),
    ("Aladdin", False),
    ("Hercules", False),
    ("Pocahontas", False),
    ("Anastasia", False),

    # EDGE CASES - Short titles
    ("Up", False),
    ("Her", False),
    ("It", False),
    ("Us", False),
    ("Jaws", False),
    ("Saw", False),
    ("300", False),
    ("9", False),
    ("Pi", False),
    ("Go", False),

    # EDGE CASES - Numbers
    ("2001 A Space Odyssey", False),
    ("1917", False),
    ("1984", False),
    ("12 Monkeys", False),
    ("21 Jump Street", False),
    ("50 First Dates", False),
    ("127 Hours", False),
    ("28 Days Later", False),
    ("48 Hours", False),
    ("8 Mile", False),
]

print(f"Total test cases: {len(test_cases)}")

Total test cases: 334


In [45]:
correct = 0

print(f"\n{'Text':30} {'Prob_RU':10} {'Prob_EN':10} {'Expected':10} {'Got':10} {'Status':8}")
print("-" * 85)

for text, expected_ru in test_cases:
    prob_ru = model_ru.probability(text)
    prob_en = model_en.probability(text)

    is_russian = prob_ru > prob_en

    status = "✅" if is_russian == expected_ru else "❌"
    if is_russian == expected_ru:
        correct += 1

    exp_str = "RU" if expected_ru else "EN"
    got_str = "RU" if is_russian else "EN"
    print(f"{text:30} {prob_ru:10.2f} {prob_en:10.2f} {exp_str:10} {got_str:10} {status:8}")

print(f"\nAccuracy: {correct}/{len(test_cases)} ({100*correct/len(test_cases):.0f}%)")


Text                           Prob_RU    Prob_EN    Expected   Got        Status  
-------------------------------------------------------------------------------------
Brat                                -0.65      -0.91 RU         RU         ✅       
Brat 2                              -1.10      -1.46 RU         RU         ✅       
Stalker                             -1.41      -0.97 RU         EN         ❌       
Solaris                             -1.88      -1.33 RU         EN         ❌       
Moskva slezam ne verit              -2.00      -2.67 RU         RU         ✅       
Leviafan                            -2.16      -2.34 RU         RU         ✅       
Nochnoy dozor                       -1.54      -2.37 RU         RU         ✅       
Dnevnoy dozor                       -1.69      -2.98 RU         RU         ✅       
Ironia sudby                        -2.09      -2.11 RU         RU         ✅       
Brilliantovaya ruka                 -1.54      -2.27 RU         RU       

## Download

In [46]:
files.download('jfmo_russian_model.pkl')
files.download('jfmo_english_model.pkl')

files.download('media_transliterated.csv')

print("Files downloaded")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloaded
